<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Application-Deployment/blob/main/20261120/netflix-shows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[netflix_title](https://www.kaggle.com/datasets/shivamb/netflix-shows)

In [1]:
import pandas as pd

# ZIP 파일로부터 CSV 직접 로드
df = pd.read_csv('/content/netflix_titles.csv.zip')
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [2]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([
        {
            'Column': col,
            'N_Unique': df[col].nunique(),
            'Dtype': df[col].dtype,
            'Null_Count': df[col].isna().sum(),
            'Null_Ratio(%)': round(df[col].isna().mean() * 100, 2),
            'Sample_Values': df[col].dropna().unique()[:7].tolist()
        }
        for col in df.columns
    ])

summarize(df)

,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values
0,show_id,8807,object,0,0.00,"[s1, s2, s3, s4, s5, s6, s7]"
1,type,2,object,0,0.00,"[Movie, TV Show]"
2,title,8807,object,0,0.00,"[Dick Johnson Is Dead, Blood & Water, Gangland..."
3,director,4528,object,2634,29.91,"[Kirsten Johnson, Julien Leclercq, Mike Flanag..."
4,cast,7692,object,825,9.37,"[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba..."
5,country,748,object,831,9.44,"[United States, South Africa, India, United St..."
6,date_added,1767,object,10,0.11,"[September 25, 2021, September 24, 2021, Septe..."
7,release_year,74,int64,0,0.00,"[2020, 2021, 1993, 2018, 1996, 1998, 1997]"
8,rating,17,object,4,0.05,"[PG-13, TV-MA, PG, TV-14, TV-PG, TV-Y, TV-Y7]"
9,duration,220,object,3,0.03,"[90 min, 2 Seasons, 1 Season, 91 min, 125 min,..."


In [17]:
from google.colab import userdata
import os

YOUR_API_KEY = userdata.get('Gemini-API-Key')
os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

In [20]:
!pip install -q google-generativeai

print("사용 가능한 Gemini 모델 목록:")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"  Model Name: {m.name}, Supported Methods: {m.supported_generation_methods}")
except Exception as e:
    print(f"모델 목록을 가져오는 중 오류가 발생했습니다. API 키를 다시 확인해 주세요.\n오류 내용: {e}")

사용 가능한 Gemini 모델 목록:
  Model Name: models/gemini-2.5-flash, Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Model Name: models/gemini-2.5-pro, Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Model Name: models/gemini-2.5-flash-preview-tts, Supported Methods: ['countTokens', 'generateContent']
  Model Name: models/gemini-2.5-pro-preview-tts, Supported Methods: ['countTokens', 'generateContent', 'batchGenerateContent']
  Model Name: models/gemma-4-26b-a4b-it, Supported Methods: ['generateContent', 'countTokens']
  Model Name: models/gemma-4-31b-it, Supported Methods: ['generateContent', 'countTokens']
  Model Name: models/gemini-flash-latest, Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
  Model Name: models/gemini-flash-lite-latest, Supported Methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateCo

In [13]:
df['description'][:10]

,description
0,"As her father nears the end of his life, filmm..."
1,"After crossing paths at a party, a Cape Town t..."
2,To protect his family from a powerful drug lor...
3,"Feuds, flirtations and toilet talk go down amo..."
4,In a city of coaching centers known to train I...
5,The arrival of a charismatic young priest brin...
6,Equestria's divided. But a bright-eyed hero be...
7,"On a photo shoot in Ghana, an American model s..."
8,A talented batch of amateur bakers face off in...
9,A woman adjusting to life after a loss contend...


In [23]:
!pip install -q langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
response = model.invoke("The sky is")
print(response.content)

[{'type': 'text', 'text': "blue during a clear day due to a phenomenon called **Rayleigh scattering**. Sunlight contains all the colors of the rainbow, but the gases in Earth's atmosphere scatter short-wavelength light (like blue and violet) in all directions much more than longer wavelengths (like red and yellow). Because our eyes are more sensitive to blue light than violet, the sky appears blue to us.\n\nDepending on what you were looking for, here are a few other ways to answer:\n\n* **At sunset or sunrise:** The sky turns red, orange, and pink because the sunlight travels through a thicker layer of atmosphere, scattering away the blue light and allowing the longer red wavelengths to pass through.\n* **At night:** The sky appears black because there is no direct sunlight illuminating the atmosphere, exposing the darkness of outer space punctuated by stars and planets.\n* **Literarily/Poetically:** The sky is often described as a canvas, a ceiling to the world, or an ever-changing m

In [24]:
print(response.text)

blue during a clear day due to a phenomenon called **Rayleigh scattering**. Sunlight contains all the colors of the rainbow, but the gases in Earth's atmosphere scatter short-wavelength light (like blue and violet) in all directions much more than longer wavelengths (like red and yellow). Because our eyes are more sensitive to blue light than violet, the sky appears blue to us.

Depending on what you were looking for, here are a few other ways to answer:

* **At sunset or sunrise:** The sky turns red, orange, and pink because the sunlight travels through a thicker layer of atmosphere, scattering away the blue light and allowing the longer red wavelengths to pass through.
* **At night:** The sky appears black because there is no direct sunlight illuminating the atmosphere, exposing the darkness of outer space punctuated by stars and planets.
* **Literarily/Poetically:** The sky is often described as a canvas, a ceiling to the world, or an ever-changing mirror of the Earth's weather and 

In [14]:
for content in df['description'][:10]:
    print(len(content))
    print(content, end="\n\n")

152
As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.

147
After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.

146
To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war.

147
Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series.

152
In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life.

152
The arrival of a charismatic young priest brings glorious miracles, ominous mysteries and renewed religious fervor to a dying town desperate to believe.

153
Equestria's divided. But a bright-eyed hero believes Earth Ponie

In [27]:
import os
import time
import pandas as pd
from google import genai
from google.genai import types

# 1) 키를 환경변수로 옮기기 (코드에 키 값이 없음)
# Colab이라면: 왼쪽 열쇠 아이콘에 GEMINI_API_KEY 저장 + "노트북 액세스" 켜기
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("Gemini-API-Key")
    os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

except ImportError:
    pass  # 로컬이면 터미널에서 미리 export 한 환경변수를 그대로 사용

# 2) 인자 없이 client 생성 (GEMINI_API_KEY를 자동으로 읽음)
client = genai.Client()
MODEL = "gemini-3.5-flash"  # 404가 나면 client.models.list()로 실제 모델명 확인

# 3) 반복문 전에 한 번만 테스트: 키/모델 문제면 여기서 바로 멈춤
client.models.generate_content(model=MODEL, contents="ping")
print("연결 OK")

def classify_genre(description):
    if pd.isna(description) or not str(description).strip():
        return "Unknown"

    prompt = f"""
    다음 영상/작품의 줄거리(description)를 읽고 가장 적절한 장르 1~2개를 영어로 짧게 답해줘.
    부가 설명 없이 장르명만 출력해. (예: Drama, Thriller / Documentary / Reality TV / Animation, Family)

    줄거리: {description}
    """
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()

sample_df = df.head().copy()
genres = []
for idx, row in sample_df.iterrows():
    try:
        genre = classify_genre(row["description"])
    except Exception as e:
        print(f"[{idx}번 행] 에러: {e}")
        genre = "Error"
    genres.append(genre)
    print(f"[{idx}번 행] 장르: {genre} | 줄거리: {str(row['description'])[:40]}...")
    time.sleep(1)

sample_df["genre"] = genres
sample_df[["description", "genre"]]

연결 OK
[0번 행] 장르: Documentary, Comedy | 줄거리: As her father nears the end of his life,...
[1번 행] 장르: Drama, Mystery | 줄거리: After crossing paths at a party, a Cape ...
[2번 행] 장르: Action, Crime | 줄거리: To protect his family from a powerful dr...
[3번 행] 장르: Reality TV, Crime | 줄거리: Feuds, flirtations and toilet talk go do...
[4번 행] 장르: Drama, Comedy | 줄거리: In a city of coaching centers known to t...
[5번 행] 장르: Horror, Mystery | 줄거리: The arrival of a charismatic young pries...
[6번 행] 장르: Animation, Family | 줄거리: Equestria's divided. But a bright-eyed h...
[7번 행] 에러: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[7번 행] 장르: Error | 줄거리: On a photo shoot in Ghana, an American m...
[8번 행] 장르: Reality TV, Game Show | 줄거리: A talented batch of amateur bakers face ...
[9번 행] 장르: Drama, Comedy | 줄거리: A woman adjusting to life after a loss c...


,description,genre
0,"As her father nears the end of his life, filmm...","Documentary, Comedy"
1,"After crossing paths at a party, a Cape Town t...","Drama, Mystery"
2,To protect his family from a powerful drug lor...,"Action, Crime"
3,"Feuds, flirtations and toilet talk go down amo...","Reality TV, Crime"
4,In a city of coaching centers known to train I...,"Drama, Comedy"
5,The arrival of a charismatic young priest brin...,"Horror, Mystery"
6,Equestria's divided. But a bright-eyed hero be...,"Animation, Family"
7,"On a photo shoot in Ghana, an American model s...",Error
8,A talented batch of amateur bakers face off in...,"Reality TV, Game Show"
9,A woman adjusting to life after a loss contend...,"Drama, Comedy"
